# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# Import the necessary libs
import os
import json

from dotenv import load_dotenv

import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
# Load environment variables (reads project/starter/.env)
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# Shared clients used by the tools below.
#
# IMPORTANT: reuse the SAME embedding function as Part 1, otherwise the query
# embeddings won't match the vectors stored in the collection.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY
)

# Connect to the persistent collection created in Part 1
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

# Tavily client for web search
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str) -> list[dict]:
    """Semantic search: Finds the most relevant games in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=5)

    games = []
    for doc_id, meta, dist in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        games.append({
            "id": doc_id,
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "Description": meta.get("Description"),
            # distance -> rough similarity score (higher is more similar)
            "similarity": round(1 - dist, 4),
        })
    return games

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    """Structured result of judging whether retrieved docs answer the question."""
    useful: bool
    description: str


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> dict:
    """Based on the user's question and the list of retrieved documents,
    analyze the usability of the documents to respond to that question.

    args:
    - question: original question from the user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector DB

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    # Use an LLM as a judge, returning a structured EvaluationReport
    judge = LLM(model="gpt-4o-mini", temperature=0)
    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond to the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )),
        UserMessage(content=(
            f"Question: {question}\n\n"
            f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2)}"
        )),
    ]
    response = judge.invoke(messages, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    return report.model_dump()

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> dict:
    """Web search: Finds information on the web about the game industry.

    Use this when the internal vector DB does not contain enough information
    to answer the question (as judged by `evaluate_retrieval`).

    args:
    - question: a question about the game industry.
    """
    response = tavily_client.search(
        question,
        max_results=5,
        include_answer=True,
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content"),
            }
            for r in response.get("results", [])
        ],
    }

### Agent

In [ ]:
# Create the UdaPlay agent using the StateMachine-based Agent abstraction (lib/agents.py)
instructions = (
    "You are UdaPlay, an AI research assistant for the video game industry. "
    "You answer questions about video games such as release dates, platforms, "
    "genres, publishers, and descriptions.\n\n"
    "Follow this process for every question:\n"
    "1. Call `retrieve_game` to search the internal knowledge base first.\n"
    "2. Call `evaluate_retrieval`, passing the original question and the documents "
    "returned by `retrieve_game`, to decide whether they are sufficient.\n"
    "3. If (and only if) the retrieved documents are NOT useful, call `game_web_search` "
    "to look the answer up on the web.\n"
    "4. Give a clear, concise final answer. Include the year and platform when relevant, "
    "and state whether the answer came from the internal knowledge base or a web search. "
    "If you cannot find a reliable answer, say so instead of guessing."
)

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

In [ ]:
# Invoke the agent on the sample questions.
# Each question uses its own session_id so the runs are independent.
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, question in enumerate(questions):
    run = agent.invoke(question, session_id=f"q{i}")
    final_state = run.get_final_state()
    answer = final_state["messages"][-1].content

    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"(tokens used: {final_state.get('total_tokens')})")
    print("-" * 80)

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes